<a href="https://colab.research.google.com/github/elinimuleg00-bot/30-Days-of-Python-DevOps/blob/main/Day09/Day9_log_parser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Log Parser & Pattern Extractor
A DevOps log-parsing utility demonstrating regular expressions (re.findall) for IPv4 and log-level extraction, string manipulation (strip, split), stream processing, and script reusability via if __name__ == "__main__"

In [ ]:
import os
import re

#stores the filename of our log file in a variable using uppercase - indicating to other developers that its a constant configuration value
LOG_FILE_PATH = "server_logs.txt"
#creating a multi-line string
sample_logs = """
2026-09-04 08:20:00 [INFO] User login success from 192.168.1.100
2026-09-04 08:21:15 [ERROR] Failed login attempt from 10.0.0.45 on port 8080
2026-09-04 08:22:01 [WARNING] High disk usage on server-01
2026-09-04 08:23:40 [ERROR] Unauthorized root access attempt from 172.16.254.1
"""

with open(LOG_FILE_PATH,'w') as file: # opening the log file in write mode
  file.write(sample_logs)

IP_PATTERN = r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}"  # matches IP addresses
LOG_LEVEL_PATTERN = r"\[(INFO|WARNING|ERROR)\]" # looks for actual [ ] and any words containing like INFO,ERROR

def parse_logs(file_path):
  """Reads a log file line-by-line, cleans whitespace, and extracts metadata using regex."""

  extracted_ips = []  # an empty list to collect all IP addresses
  error_count = 0 # an int counter to keep track of how many [ERROR] logs detected

  try:
    with open(file_path, "r") as file:  # Opens server_logs.txt in read-only mode
      print("---LOG ANALYSIS REPORT---")

      for line in file: # loops through the log file one line at a time -> stream processing!
        clean_line = line.strip() # String Method: Remove leading/trailing spaces and newline characters (\n)

        if not clean_line:  # skip if line is empty
          continue

        ips = re.findall(IP_PATTERN, clean_line) # a regex that extracts IPs from current line and returns them as a list
        if ips: # check if any IPs were found on this line
          extracted_ips.extend(ips) # adds those found IPs into extracted_ips list

        log_levels = re.findall(LOG_LEVEL_PATTERN, clean_line)
        if "ERROR" in log_levels: # if the string error is present in the result, increment by 1
          error_count += 1
          parts = clean_line.split("]") # Splits the log line at the ] character into a list of strings.
          # strips everything after ] , Checks if the line actually contained a ] character and split successfully into at least 2 pieces and strips. if not len > 1 uses the whole raw line
          message = parts[-1].strip() if len(parts) > 1 else clean_line
          print(f"CRITICAL ERROR DETECTED: {message}")

    print("\n---SUMMARY---")
    print(f"Total IP Addresses Extracted: {len(extracted_ips)}")
    print(f"Unique IPs: {set(extracted_ips)}") # Converts the list into a set to automatically remove duplicate IPs and show only unique values.
    print(f"Total errors Found: {error_count}")

  except FileNotFoundError:
    print(f"Error: The file'{file_path}' was not found.")
  except Exception as e:
    print(f"An unexpected error occured: {e}")

# safeguard for ur python code, quietly imports the parse_logs function into memory without running it. You can then call whenever & wherever u want
# Every Python file has a hidden internal variable called __name__ :
# 1. When u run the file directly, checks if the condition is true(Python automatically sets __name__ = "__main__") and runs parse logs
# 2. When another script imports ur file (e.g import log_parser inside Day15 script) python sets __name__="log_parser"(the name of the file,not __main__)
#     just borrowing its tools! I will NOT auto-run parse_logs(). I will just hand over the tool quietly until called
if __name__ == "__main__":
  parse_logs(LOG_FILE_PATH)


---LOG ANALYSIS REPORT---
CRITICAL ERROR DETECTED: Failed login attempt from 10.0.0.45 on port 8080
CRITICAL ERROR DETECTED: Unauthorized root access attempt from 172.16.254.1

---SUMMARY---
Total IP Addresses Extracted: 3
Unique IPs: {'172.16.254.1', '192.168.1.100', '10.0.0.45'}
Total errors Found: 2
